# 07 - Stochastic Variational Inference (SVI)

For large datasets, computing the ELBO over the full dataset at every step is expensive.
**Stochastic Variational Inference** uses minibatches so optimization scales better.

This notebook covers:
1. Why full-batch VI becomes expensive
2. The minibatch ELBO idea
3. Noisy but useful gradient estimates
4. A toy demonstration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: Full-data ELBO

For dataset $x_{1:N}$, the ELBO often looks like
$$
athcal{L} = um_{i=1}^N athbb{E}_{q}[og p(x_i id z_i, 	heta)] - athrm{KL}(q  p).
$$

When $N$ is huge, evaluating the full sum each step is too slow.

In [ ]:
N = 5000
true_mean = 2.0
sigma_obs = 1.0
data = np.random.normal(true_mean, sigma_obs, size=N)

plt.figure(figsize=(8, 4))
plt.hist(data, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(np.mean(data), color='red', linestyle='--', label='sample mean')
plt.title('Large dataset for VI')
plt.xlabel('x')
plt.ylabel('count')
plt.legend()
plt.show()

## Part 2: Minibatch estimator

If we sample a minibatch $B$ of size $M$, then
$$
um_{i=1}^N og p(x_i id 	heta) pprox rac{N}{M} um_{i n B} og p(x_i id 	heta).
$$

This gives a noisy but unbiased estimator of the full-data contribution.
That is the key scaling trick behind SVI.

In [ ]:
def full_data_loglik(mu):
    return np.sum(stats.norm.logpdf(data, loc=mu, scale=sigma_obs))

def minibatch_loglik(mu, batch_size=100):
    idx = np.random.choice(N, size=batch_size, replace=False)
    batch = data[idx]
    return (N / batch_size) * np.sum(stats.norm.logpdf(batch, loc=mu, scale=sigma_obs))

mu_grid = np.linspace(1.4, 2.6, 60)
full_vals = np.array([full_data_loglik(mu) for mu in mu_grid])
mini_vals_1 = np.array([minibatch_loglik(mu, batch_size=100) for mu in mu_grid])
mini_vals_2 = np.array([minibatch_loglik(mu, batch_size=100) for mu in mu_grid])

plt.figure(figsize=(9, 4))
plt.plot(mu_grid, full_vals, label='full-data log-likelihood', linewidth=3, color='navy')
plt.plot(mu_grid, mini_vals_1, label='minibatch estimate 1', linewidth=2, color='darkorange', alpha=0.9)
plt.plot(mu_grid, mini_vals_2, label='minibatch estimate 2', linewidth=2, color='seagreen', alpha=0.9)
plt.xlabel('mu')
plt.ylabel('objective contribution')
plt.title('Full objective vs minibatch approximations')
plt.legend()
plt.show()

## Part 3: Why noisy gradients are still useful

Optimization does not require a perfect gradient every time.
It only needs steps that are informative on average.

This is the same idea behind stochastic gradient descent.
SVI combines that optimization idea with variational inference.

In [ ]:
# Toy stochastic optimization of the posterior mean for a Gaussian model
prior_mean = 0.0
prior_std = 3.0
lr = 0.01
n_steps = 300
batch_size = 64
mu_est = -1.5
trajectory = [mu_est]

for _ in range(n_steps):
    idx = np.random.choice(N, size=batch_size, replace=False)
    batch = data[idx]

    # Gradient of scaled minibatch log posterior wrt mu
    grad_lik = (N / batch_size) * np.sum((batch - mu_est) / (sigma_obs**2))
    grad_prior = (prior_mean - mu_est) / (prior_std**2)
    grad = grad_lik + grad_prior

    mu_est = mu_est + lr * grad / N
    trajectory.append(mu_est)

plt.figure(figsize=(8, 4))
plt.plot(trajectory, color='purple', linewidth=2)
plt.axhline(np.mean(data), color='black', linestyle='--', label='sample mean')
plt.title('Stochastic updates using minibatches')
plt.xlabel('step')
plt.ylabel('mu estimate')
plt.legend()
plt.show()

print(f'Final stochastic estimate: {trajectory[-1]:.3f}')
print(f'Data mean: {np.mean(data):.3f}')

## Part 4: SVI in modern models

In practice, SVI is used when:
- the dataset is large
- the variational family is parameterized by neural networks
- the ELBO is optimized with minibatches

This is exactly the large-scale setting for VAEs and many deep latent-variable models.

## Summary

What to remember:
1. Full-batch VI can be too expensive on large datasets
2. SVI replaces full sums with minibatch estimates
3. The resulting gradients are noisy but useful
4. SVI is the practical bridge between VI theory and large-scale learning

In [ ]:
# Exercises
# 1) Change the minibatch size and see how noisy the objective becomes.
# 2) Try different learning rates in the stochastic update loop.
# 3) Increase N and think about why full-batch evaluation scales poorly.

pass